# 03 · Model Explainability — GradCAM & Attention Analysis
Explains *why* the model makes each decision:
- **Thermal GradCAM** — where the IR camera contributes most
- **Visual GradCAM** — which surface region per section is critical
- **Cross-section attention** — left / mid / right importance
- **Modality token weights** — which of the 4 modalities dominates
- **TCN channel saliency** — which DXP signal drives the prediction
- **ROI temperature importance** — which SKZ measurement matters

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, str(Path.cwd().parent))
import config as C
from data.loaders    import load_parquet, build_cv_index, join_cv_to_parquet, load_tabular_vector
from data.dataset    import MultiModalDataset, collate_fn, get_tabular_cols, get_dxp_cols
from models.fusion   import InjectionMouldingFusionModel
from training.splits import build_experiment_label_map, random_split
from analysis.explainability import (
    thermal_gradcam, visual_gradcam, get_modality_weights,
    tcn_channel_saliency, roi_importance,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
print("Device:", DEVICE)

In [ ]:
# ── Load data + model ────────────────────────────────────────
df     = load_parquet(C.PARQUET_PATH)
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)

tabular_cols  = get_tabular_cols(df)
dxp_cols      = get_dxp_cols(df)
exp_label_map = build_experiment_label_map(df)

scaler   = StandardScaler()
tab_vals = [load_tabular_vector(r, tabular_cols) for _, r in df.iterrows()]
scaler.fit(np.array(tab_vals))
tab_med  = {c: float(df[c].median()) for c in tabular_cols if c in df.columns}
roi_med  = {c: float(df[c].median()) for c in C.THERMAL_ROI_COLS if c in df.columns}

ds  = MultiModalDataset(df, tabular_cols, dxp_cols, train=False,
                        tabular_medians=tab_med, roi_medians=roi_med, tabular_scaler=scaler)
ld  = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collate_fn, num_workers=0)

model = InjectionMouldingFusionModel(
    tabular_in_dim=len(tabular_cols),
    n_dxp_channels=len(dxp_cols),
    n_experiments=len(exp_label_map),
).to(DEVICE)
ckpt = C.MODELS_DIR / "fusion_final.pt"
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print("Model loaded")
else:
    print(f"WARNING: No checkpoint at {ckpt} — model is random (for layout demo only)")
model.eval()

## 1 · Select Sample Parts

In [ ]:
# Grab 1 NOK and 1 OK part for comparison
sample_batches = {}
label_idx = C.LABEL_COLS.index(C.PRIMARY_TARGET)

for batch in ld:
    lbl = batch["labels"][0, label_idx].item()
    key = "NOK" if lbl == 1 else "OK"
    if key not in sample_batches:
        sample_batches[key] = batch
    if len(sample_batches) == 2:
        break

print("Collected samples:", list(sample_batches.keys()))
for key, b in sample_batches.items():
    cycle = b.get("cycle_id", ["?"])[0]
    has_t = b["thermal_valid"][0].item()
    has_v = b["visual_valid"][0].item()
    has_s = b["sequence_valid"][0].item()
    print(f"  {key:4s} | cycle={cycle} | thermal={has_t} | visual={has_v} | seq={has_s}")

## 2 · Modality Token Weights

In [ ]:
MODALITY_NAMES = ["Thermal", "Visual", "Sequence", "Tabular"]
COLOURS        = ["#E91E63", "#2196F3", "#4CAF50", "#FF9800"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (key, batch) in zip(axes, sample_batches.items()):
    weights = get_modality_weights(model, batch, DEVICE)
    bars = ax.bar(MODALITY_NAMES, weights, color=COLOURS, edgecolor="white", linewidth=1.2)
    for bar, w in zip(bars, weights):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{w:.2f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylim(0, 1.0); ax.set_title(f"Modality Weights — {key} Part")
    ax.set_ylabel("Attention Weight")
    ax.set_xlabel("Modality")

plt.suptitle("Cross-Modal Fusion: Which Modality Drives the Decision?", fontsize=12)
plt.tight_layout(); plt.show()

## 3 · Thermal GradCAM

In [ ]:
import matplotlib.cm as cm

def _overlay(base, cam, alpha=0.5):
    base_n = (base - base.min()) / (base.max() - base.min() + 1e-6)
    if base_n.dtype != np.uint8:
        base_uint = (base_n * 255).astype(np.uint8)
    rgb = np.stack([base_uint]*3, axis=2)
    heat = (cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
    return (alpha * heat + (1 - alpha) * rgb).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row_i, (key, batch) in enumerate(sample_batches.items()):
    if not batch["thermal_valid"][0].item():
        for col in range(3):
            axes[row_i, col].text(0.5, 0.5, "No thermal data", ha="center",
                                  transform=axes[row_i, col].transAxes)
        continue

    imgs = batch["thermal_img"][0].cpu().numpy()   # (3, 480, 640)
    f1, f2 = imgs[0], imgs[1]

    try:
        cam_map = thermal_gradcam(model, batch, DEVICE, label_idx=label_idx)
        overlay = _overlay(f1, cam_map)
    except Exception as e:
        cam_map = np.zeros((480, 640)); overlay = np.zeros((480, 640, 3), dtype=np.uint8)
        print(f"GradCAM error ({key}):", e)

    def _norm(x):
        return (x - x.min()) / (x.max() - x.min() + 1e-6)

    axes[row_i, 0].imshow(_norm(f1), cmap="inferno")
    axes[row_i, 0].set_title(f"{key} — Frame 1 (filling)")
    axes[row_i, 1].imshow(_norm(f2), cmap="inferno")
    axes[row_i, 1].set_title(f"{key} — Frame 2 (cooling)")
    axes[row_i, 2].imshow(overlay)
    axes[row_i, 2].set_title(f"{key} — GradCAM overlay")

    for ax in axes[row_i]: ax.axis("off")

plt.suptitle("Thermal GradCAM: Where the IR encoder attends (LBL_NOK)", fontsize=13)
plt.tight_layout(); plt.savefig(C.PLOTS_DIR/"thermal_gradcam.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · TCN Channel Saliency & ROI Importance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for col_i, (key, batch) in enumerate(sample_batches.items()):
    # TCN channel saliency
    if batch["sequence_valid"][0].item():
        try:
            sal = tcn_channel_saliency(model, batch, DEVICE, label_idx=label_idx)
        except Exception as e:
            sal = np.ones(len(dxp_cols)) / len(dxp_cols)
            print("TCN saliency error:", e)
    else:
        sal = np.zeros(len(dxp_cols))

    ch_names = [c.replace("DXP_","") for c in dxp_cols]
    bar_cols  = ["#E53935" if s == sal.max() else "#64B5F6" for s in sal]
    axes[0, col_i].bar(range(len(sal)), sal, color=bar_cols, edgecolor="white")
    axes[0, col_i].set_xticks(range(len(sal)))
    axes[0, col_i].set_xticklabels(ch_names, rotation=35, ha="right", fontsize=8)
    axes[0, col_i].set_title(f"TCN Saliency — {key}")
    axes[0, col_i].set_ylabel("Normalised gradient importance")

    # ROI importance
    if batch["thermal_valid"][0].item():
        try:
            roi_imp = roi_importance(model, batch, DEVICE, label_idx=label_idx)
        except Exception as e:
            roi_imp = np.ones(10) / 10
            print("ROI error:", e)
    else:
        roi_imp = np.zeros(10)

    roi_names = [c.replace("IR_","") for c in C.THERMAL_ROI_COLS]
    axes[1, col_i].bar(range(len(roi_imp)), roi_imp,
                       color=["#EF5350" if v==roi_imp.max() else "#A5D6A7" for v in roi_imp])
    axes[1, col_i].set_xticks(range(len(roi_imp)))
    axes[1, col_i].set_xticklabels(roi_names, rotation=35, ha="right", fontsize=8)
    axes[1, col_i].set_title(f"ROI Temperature Importance — {key}")
    axes[1, col_i].set_ylabel("Gradient importance")

plt.suptitle("Feature Importance: DXP Channels & Thermal ROI Temperatures", fontsize=12)
plt.tight_layout()
plt.savefig(C.PLOTS_DIR/"channel_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 5 · Multi-Part Modality Weight Heatmap

In [ ]:
import numpy as np
weight_rows, part_ids, labels_list = [], [], []

for i, batch in enumerate(ld):
    if i >= 40:
        break
    weights = get_modality_weights(model, batch, DEVICE)
    weight_rows.append(weights)
    part_ids.append(str(batch.get("cycle_id", [i])[0]))
    labels_list.append(int(batch["labels"][0, label_idx].item()))

W = np.array(weight_rows)   # (N, 4)

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(W, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(4))
ax.set_xticklabels(["Thermal", "Visual", "Sequence", "Tabular"])
ax.set_yticks(range(len(part_ids)))
ax.set_yticklabels(
    [f"{'★' if l else '·'} {pid}" for pid, l in zip(part_ids, labels_list)],
    fontsize=7
)
ax.set_xlabel("Modality")
ax.set_ylabel("Part (★=NOK)")
plt.colorbar(im, ax=ax, label="Attention weight")
ax.set_title("Per-Part Modality Weights\n(shows adaptive fusion behaviour)", fontsize=11)
plt.tight_layout()
plt.savefig(C.PLOTS_DIR/"modality_weight_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Higher weight on Thermal for thermally-driven defects,")
print("Higher weight on Sequence for pressure-driven defects (sink marks, underfill)")